# **Homework 3 - Convolutional Neural Network**

此為李宏毅老師機器學習課程作業3的程式碼，以官方範例為基礎修改。

作業目標：
- **Easy**：建立基本 CNN，跑過 Simple Baseline (44.862%)
- **Medium**：加入 Data Augmentation，跑過 Medium Baseline (52.807%)
- **Hard**：實作 Semi-supervised Learning（Pseudo-label），跑過 Strong Baseline (82.138%)

## **Step 1: 下載資料集**

In [ ]:
# 從 Google Drive 下載資料集
!gdown --id '1awF7pZ9Dz7X1jn1_QAiKN-_v56veCEKy' --output food-11.zip

# 解壓縮（-q 代表安靜模式，不顯示每個檔案）
!unzip -q food-11.zip

## **Step 2: 匯入套件**

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
# ConcatDataset：合併多個 Dataset
# Subset：從 Dataset 中取部分資料
from torch.utils.data import ConcatDataset, DataLoader, Subset, Dataset
from torchvision.datasets import DatasetFolder
from tqdm.auto import tqdm

## **Step 3: 資料前處理與 Data Augmentation**

### 為什麼要做 Data Augmentation？
訓練資料只有 3080 張（280×11），相當少。透過對圖片做隨機變換，
讓模型看到同一張圖的不同「版本」，等同於擴充訓練資料量，
可以有效降低 Overfitting。

### 哪些 Augmentation 對食物辨識有用？
- `RandomHorizontalFlip`：水平翻轉（食物不分左右）
- `RandomRotation`：小角度旋轉（拍攝角度不同）
- `ColorJitter`：亮度/對比/飽和度變化（光線環境不同）
- `RandomResizedCrop`：隨機裁切後縮放（模擬遠近不同）

### 注意：Validation/Test 不做 Augmentation
Augmentation 只用於訓練。驗證和測試要用原始圖片評估真實效能。

In [ ]:
# 訓練用的 Transform（含 Data Augmentation）
train_tfm = transforms.Compose([
    # 先隨機裁切再縮放到 128x128，模擬不同拍攝距離
    # scale=(0.7, 1.0) 代表裁切後面積為原圖的 70%~100%
    transforms.RandomResizedCrop(128, scale=(0.7, 1.0)),

    # 以 50% 機率水平翻轉（食物辨識不受左右影響）
    transforms.RandomHorizontalFlip(p=0.5),

    # 隨機旋轉 ±15 度（拍攝角度稍微歪斜）
    transforms.RandomRotation(15),

    # 隨機調整亮度、對比度、飽和度（光線環境差異）
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),

    # 轉成 PyTorch Tensor，並自動將像素值從 [0, 255] 縮放到 [0.0, 1.0]
    # ToTensor() 必須放在最後
    transforms.ToTensor(),

    # Normalize：以 ImageNet 常用的 mean/std 正規化
    # 讓每個 channel 的分布接近標準常態，有助於訓練穩定
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# 驗證/測試用的 Transform（不做 Augmentation，只做必要的前處理）
test_tfm = transforms.Compose([
    # 直接縮放到 128x128（不裁切，保留完整圖片）
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

## **Step 4: 載入資料集**

`DatasetFolder` 會根據子資料夾名稱自動分配 class label，
例如 `food-11/training/labeled/0/` 裡的圖片 label 就是 0。

Unlabeled data 只有一個假資料夾 `0`，所以它的 label 都是 0（無意義），
之後我們會用 Pseudo-label 覆蓋。

In [ ]:
# Batch size：每次送進模型的圖片數量
# 越大梯度越穩定，但佔用 GPU 記憶體也越多
batch_size = 128

# 建立各個 Dataset
# loader：告訴 DatasetFolder 如何讀取圖片（用 PIL 的 Image.open）
# extensions：只讀取 .jpg 檔案
train_set = DatasetFolder(
    "food-11/training/labeled",
    loader=lambda x: Image.open(x).convert("RGB"),
    extensions="jpg",
    transform=train_tfm
)
valid_set = DatasetFolder(
    "food-11/validation",
    loader=lambda x: Image.open(x).convert("RGB"),
    extensions="jpg",
    transform=test_tfm
)
# Unlabeled data：用 train_tfm（含 Augmentation），因為訓練時會用到
unlabeled_set = DatasetFolder(
    "food-11/training/unlabeled",
    loader=lambda x: Image.open(x).convert("RGB"),
    extensions="jpg",
    transform=train_tfm
)
test_set = DatasetFolder(
    "food-11/testing",
    loader=lambda x: Image.open(x).convert("RGB"),
    extensions="jpg",
    transform=test_tfm
)

# 建立 DataLoader
# num_workers=0：Colab 的 multiprocessing 環境容易出錯，設 0 用主 thread 讀取
# pin_memory=True：將資料鎖定在記憶體，加快 CPU→GPU 的傳輸
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=0)


## **Step 5: 模型架構**

### CNN 的運作原理
- **Conv2d**：用一個小的 kernel（例如 3×3）在圖片上滑動，提取局部特徵
  - 前幾層偵測邊緣、顏色等低階特徵
  - 後幾層偵測形狀、紋理等高階特徵
- **BatchNorm2d**：對每個 channel 做正規化，穩定訓練
- **ReLU**：非線性激活函數，`max(0, x)`
- **MaxPool2d**：取區域最大值，縮小 feature map 的寬高（降維）
- **Linear**：全連接層，最終輸出 11 個 class 的 logit

### 圖片尺寸的變化（以輸入 [B, 3, 128, 128] 為例）：
- Conv(3→64) + Pool(2) → [B, 64, 64, 64]
- Conv(64→128) + Pool(2) → [B, 128, 32, 32]
- Conv(128→256) + Pool(2) → [B, 256, 16, 16]
- Conv(256→512) + Pool(2) → [B, 512, 8, 8]
- Flatten → [B, 512×8×8] = [B, 32768]
- Linear → [B, 11]

### 警告：不能使用預訓練權重
若使用 ResNet 等架構，必須設定 `pretrained=False`。

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()

        # CNN 特徵提取部分
        # 每個 block：Conv → BatchNorm → ReLU → MaxPool
        # 輸入圖片尺寸：[B, 3, 128, 128]
        self.cnn_layers = nn.Sequential(
            # Block 1: [B, 3, 128, 128] → [B, 64, 64, 64]
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2: [B, 64, 64, 64] → [B, 128, 32, 32]
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3: [B, 128, 32, 32] → [B, 256, 16, 16]
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 4: [B, 256, 16, 16] → [B, 512, 8, 8]
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # 全連接分類部分
        # 輸入：512 × 8 × 8 = 32768（展平後的特徵向量）
        # 輸出：11（對應 11 個食物類別）
        self.fc_layers = nn.Sequential(
            nn.Linear(512 * 8 * 8, 1024),
            nn.ReLU(),
            # Dropout：訓練時隨機關閉 30% 的神經元，防止 Overfitting
            nn.Dropout(p=0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, 11),
        )

    def forward(self, x):
        # x 的形狀：[batch_size, 3, 128, 128]

        # 通過 CNN 層提取特徵
        x = self.cnn_layers(x)  # → [B, 512, 8, 8]

        # 展平：將 3D feature map 壓成 1D 向量
        # flatten(1) 代表從第 1 個維度開始展平（保留 batch 維度）
        x = x.flatten(1)  # → [B, 32768]

        # 通過全連接層分類
        x = self.fc_layers(x)  # → [B, 11]
        return x

## **Step 6: Pseudo-label 函數（Hard — Semi-supervised Learning）**

### 核心想法
我們有 6786 張沒有標籤的圖片。
用已訓練好的模型去預測這些圖片的類別，
如果模型夠有信心（最高機率 > threshold），
就把預測結果當作「偽標籤（Pseudo-label）」，
把這些圖片加進訓練集繼續訓練。

### 為什麼需要自訂 PseudoDataset？
`DatasetFolder` 的 label 是根據資料夾名稱固定的，無法動態修改。
所以我們自己寫一個簡單的 Dataset 類別，
直接儲存 `(圖片 tensor, pseudo-label)` 的列表。

In [ ]:
class PseudoDataset(Dataset):
    """
    儲存帶有 Pseudo-label 的圖片資料集。

    只儲存「檔案路徑 + pseudo-label」，不預先載入圖片 tensor。
    好處：RAM 用量極低，圖片在 __getitem__ 被呼叫時才從磁碟讀取。
    """

    def __init__(self, samples, transform):
        # samples：list of (image_path, pseudo_label_int)
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        # 從磁碟讀取圖片，.convert("RGB") 確保灰階或 RGBA 圖片也能正常處理
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


def get_pseudo_labels(dataset, model, threshold=0.65):
    """
    對 unlabeled dataset 生成 Pseudo-label。

    流程：
    1. 用模型對每張圖預測，取得各 class 的機率
    2. 過濾掉最高機率低於 threshold 的圖片（模型不確定的不用）
    3. 對通過過濾的圖片，用 argmax 作為 Pseudo-label
    4. 回傳 PseudoDataset（只含高信心圖片的路徑 + label）

    Args:
        dataset: unlabeled DatasetFolder（需有 .samples 屬性）
        model: 當前已訓練的模型
        threshold: 信心門檻值，預設 0.65

    Returns:
        PseudoDataset：只含高信心圖片的新 Dataset
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # num_workers=0：Colab 的 multiprocessing 環境不穩定，設 0 最安全
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    # 切換成 eval 模式：關閉 Dropout，BatchNorm 用全局統計值
    model.eval()

    # Softmax：將 logit 轉成機率（各 class 機率加總為 1）
    softmax = nn.Softmax(dim=-1)

    # 儲存通過過濾的 (image_path, pseudo_label) 對
    pseudo_samples = []
    # 追蹤當前處理到 dataset.samples 的哪個位置
    # （因為 shuffle=False，順序與 dataset.samples 完全對應）
    sample_idx = 0

    for imgs, _ in tqdm(data_loader, desc="Generating pseudo-labels"):
        with torch.no_grad():
            logits = model(imgs.to(device))  # → [B, 11]

        probs = softmax(logits)  # → [B, 11]
        max_probs, pseudo_labels = probs.max(dim=-1)  # → [B], [B]
        confident_mask = (max_probs >= threshold).cpu()  # → [B] boolean

        for i, (label, is_confident) in enumerate(zip(pseudo_labels.cpu(), confident_mask)):
            if is_confident:
                # 從 DatasetFolder 的 .samples 取得對應的檔案路徑
                # dataset.samples 格式：[(path, class_idx), ...]
                path, _ = dataset.samples[sample_idx + i]
                pseudo_samples.append((path, label.item()))

        sample_idx += len(imgs)

    # 切回 train 模式
    model.train()

    print(f"Pseudo-labeled: {len(pseudo_samples)} / {len(dataset)} images (threshold={threshold})")
    # 回傳時傳入 train_tfm，讓 pseudo-labeled 圖片訓練時也有 Augmentation
    return PseudoDataset(pseudo_samples, train_tfm)


## **Step 7: 訓練**

### 訓練策略
- 前 **10 個 epoch** 只用 labeled data，讓模型先有基本能力再做 pseudo-labeling
- 之後每個 epoch 重新生成 pseudo-label，合併進訓練集
- 使用 **Early Stopping**：若驗證 accuracy 超過 10 個 epoch 沒有改善，提早停止
- 儲存驗證 accuracy 最高的模型（Best Model Saving）

In [ ]:
# 判斷使用 GPU 還是 CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用裝置：{device}")

# 初始化模型，並移到指定裝置（GPU/CPU）
model = Classifier().to(device)

# Loss Function：Cross-Entropy Loss
criterion = nn.CrossEntropyLoss()

# Optimizer：Adam
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)

# 訓練總 epoch 數
n_epochs = 100

# 從第幾個 epoch 開始啟用 Semi-supervised Learning
# 前 10 epoch 先只用 labeled data，讓模型有足夠能力再做 pseudo-labeling
semi_start_epoch = 10

# Pseudo-label 的信心門檻值
pseudo_threshold = 0.65

# Early Stopping 設定
patience = 10
best_val_acc = 0.0
no_improve_count = 0

# 訓練迴圈
for epoch in range(n_epochs):

    # ---------- Semi-supervised Learning ----------
    if epoch >= semi_start_epoch:
        pseudo_set = get_pseudo_labels(unlabeled_set, model, threshold=pseudo_threshold)

        if len(pseudo_set) > 0:
            concat_dataset = ConcatDataset([train_set, pseudo_set])
            train_loader = DataLoader(
                concat_dataset, batch_size=batch_size, shuffle=True,
                num_workers=0, pin_memory=True
            )
        else:
            train_loader = DataLoader(
                train_set, batch_size=batch_size, shuffle=True,
                num_workers=0, pin_memory=True
            )
    else:
        train_loader = DataLoader(
            train_set, batch_size=batch_size, shuffle=True,
            num_workers=0, pin_memory=True
        )

    # ---------- Training ----------
    model.train()

    train_loss = []
    train_accs = []

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1:03d} Train"):
        imgs, labels = imgs.to(device), labels.to(device)

        logits = model(imgs)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=10)
        optimizer.step()

        acc = (logits.argmax(dim=-1) == labels).float().mean()
        train_loss.append(loss.item())
        train_accs.append(acc.item())

    avg_train_loss = sum(train_loss) / len(train_loss)
    avg_train_acc = sum(train_accs) / len(train_accs)
    print(f"[ Train | {epoch+1:03d}/{n_epochs:03d} ] loss = {avg_train_loss:.5f}, acc = {avg_train_acc:.5f}")

    # ---------- Validation ----------
    model.eval()

    valid_loss = []
    valid_accs = []

    for imgs, labels in tqdm(valid_loader, desc=f"Epoch {epoch+1:03d} Valid"):
        imgs, labels = imgs.to(device), labels.to(device)

        with torch.no_grad():
            logits = model(imgs)

        loss = criterion(logits, labels)
        acc = (logits.argmax(dim=-1) == labels).float().mean()

        valid_loss.append(loss.item())
        valid_accs.append(acc.item())

    avg_valid_loss = sum(valid_loss) / len(valid_loss)
    avg_valid_acc = sum(valid_accs) / len(valid_accs)
    print(f"[ Valid | {epoch+1:03d}/{n_epochs:03d} ] loss = {avg_valid_loss:.5f}, acc = {avg_valid_acc:.5f}")

    # ---------- Best Model Saving & Early Stopping ----------
    if avg_valid_acc > best_val_acc:
        best_val_acc = avg_valid_acc
        no_improve_count = 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  --> 新的最佳 Validation Accuracy: {best_val_acc:.5f}，已儲存模型")
    else:
        no_improve_count += 1
        print(f"  --> 未改善 ({no_improve_count}/{patience})")
        if no_improve_count >= patience:
            print(f"Early Stopping 觸發！停止訓練。")
            break

print(f"\n訓練完成，最佳 Validation Accuracy: {best_val_acc:.5f}")


## **Step 8: 預測與輸出**

載入最佳模型，對測試集做預測，輸出符合 Kaggle 格式的 CSV 檔。

In [ ]:
# 載入最佳模型權重（驗證 accuracy 最高的那個）
model.load_state_dict(torch.load("best_model.pth"))

# 切換成 eval 模式
model.eval()

predictions = []

for imgs, _ in tqdm(test_loader, desc="Testing"):
    # 測試時不需要梯度
    with torch.no_grad():
        logits = model(imgs.to(device))  # → [B, 11]

    # argmax：取機率最高的 class 作為預測結果
    predictions.extend(logits.argmax(dim=-1).cpu().numpy().tolist())

print(f"共預測 {len(predictions)} 張圖片")

In [ ]:
# 儲存預測結果為 CSV 檔（符合 Kaggle 提交格式）
# 格式：第一行是 "Id,Category"，之後每行是 "圖片編號,預測類別"
with open("predict.csv", "w") as f:
    f.write("Id,Category\n")
    for i, pred in enumerate(predictions):
        f.write(f"{i},{pred}\n")

print("predict.csv 已儲存！共", len(predictions), "筆預測")